# Customer Intelligence Features


| Feature Type | Purpose |
|---|---|
| RFM Features | Customer value analysis |
| Behavioral Features | Customer behavior understanding |
| Aggregated Metrics | ML-ready dataset |

---

## This notebook becomes the foundation for:

- K-Means
- DBSCAN
- PCA
- Customer Personas
- Business Recommendations

In [24]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [26]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

In [27]:
df.head()

,CustomerID,ProductID,Quantity,Price,TransactionDate,PaymentMethod,StoreLocation,ProductCategory,DiscountApplied(%),TotalAmount
0,109318,C,7,80.079844,2023-12-26 12:32:00,Cash,"176 Andrew Cliffs\nBaileyfort, HI 93354",Books,18.677100,455.862764
1,993229,C,4,75.195229,2023-08-05 00:00:00,Cash,"11635 William Well Suite 809\nEast Kara, MT 19483",Home Decor,14.121365,258.306546
2,579675,A,8,31.528816,2024-03-11 18:51:00,Cash,"910 Mendez Ville Suite 909\nPort Lauraland, MO...",Books,15.943701,212.015651
3,799826,D,5,98.880218,2023-10-27 22:00:00,PayPal,"87522 Sharon Corners Suite 500\nLake Tammy, MO...",Books,6.686337,461.343769
4,121413,A,7,93.188512,2023-12-22 11:38:00,Cash,"0070 Michelle Island Suite 143\nHoland, VA 80142",Electronics,4.030096,626.030484


### Create Reference Date
RFM requires a current  reference date.

In [28]:
reference_date = df["TransactionDate"].max()
reference_date

Timestamp('2024-04-28 22:22:00')

# Create RFM Table

## Recency

Days since last purchase.

## Frequency

Total number of transactions.

## Monetary

Total customer spending.

In [29]:
rfm = df.groupby("CustomerID").agg({
    "TransactionDate": lambda x: (reference_date - x.max()).days,
    "CustomerID": "count",
    "TotalAmount": "sum"
})

In [30]:
rfm.columns = ["Recency", "Frequency", "Monetary"]

In [31]:
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
14,266,1,256.232791
42,345,1,502.656523
49,328,1,21.399047
59,27,2,249.492696
65,315,1,548.006625


### Reset Index

In [32]:
rfm = rfm.reset_index()

## BEHAVIORAL FEATURE ENGINEERING

### Average Quantity Purchased

In [33]:
avg_quantity = df.groupby("CustomerID")["Quantity"].mean().reset_index()
avg_quantity.columns = ["CustomerID", "AvgQuantity"]

### Average Discount Usage

In [34]:
avg_discount = df.groupby("CustomerID")["DiscountApplied(%)"].mean().reset_index()
avg_discount.columns = ["CustomerID", "AvgDiscount"]

### Favorite Product Category

In [35]:
favorite_category = (
    df.groupby(["CustomerID", "ProductCategory"])
    .size()
    .reset_index(name="Count")
)

favorite_category = favorite_category.loc[
    favorite_category.groupby("CustomerID")["Count"].idxmax()
]

favorite_category = favorite_category[["CustomerID", "ProductCategory"]]
favorite_category.columns = ["CustomerID", "FavoriteCategory"]

### Preferred Payment Method

In [36]:
payment_pref = (
    df.groupby(["CustomerID", "PaymentMethod"])
    .size()
    .reset_index(name="Count")
)

payment_pref = payment_pref.loc[
    payment_pref.groupby("CustomerID")["Count"].idxmax()
]

payment_pref = payment_pref[["CustomerID", "PaymentMethod"]]
payment_pref.columns = ["CustomerID", "PreferredPayment"]

### Purchase Interval
Measures buying cycle behavior.

In [37]:
df = df.sort_values(["CustomerID", "TransactionDate"])

df["PreviousPurchase"] = df.groupby("CustomerID")["TransactionDate"].shift(1)

df["PurchaseInterval"] = (
    df["TransactionDate"] - df["PreviousPurchase"]
).dt.days

### Average Purchase Interval

In [38]:
purchase_interval = (
    df.groupby("CustomerID")["PurchaseInterval"]
    .mean()
    .reset_index()
)

purchase_interval.columns = ["CustomerID", "AvgPurchaseInterval"]

### Merge All Features

In [39]:
customer_features = rfm.merge(avg_quantity, on="CustomerID")
customer_features = customer_features.merge(avg_discount, on="CustomerID")
customer_features = customer_features.merge(favorite_category, on="CustomerID")
customer_features = customer_features.merge(payment_pref, on="CustomerID")
customer_features = customer_features.merge(purchase_interval, on="CustomerID")

### Final Feature Dataset

In [40]:
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,AvgQuantity,AvgDiscount,FavoriteCategory,PreferredPayment,AvgPurchaseInterval
0,14,266,1,256.232791,5.0,15.504050,Books,Credit Card,NaN
1,42,345,1,502.656523,7.0,19.191090,Home Decor,PayPal,NaN
2,49,328,1,21.399047,1.0,14.923377,Electronics,Debit Card,NaN
3,59,27,2,249.492696,6.0,7.175628,Clothing,Debit Card,225.0
4,65,315,1,548.006625,8.0,8.319811,Clothing,PayPal,NaN


In [41]:
customer_features.shape

(95215, 9)

In [42]:
customer_features.isnull().sum()

CustomerID                 0
Recency                    0
Frequency                  0
Monetary                   0
AvgQuantity                0
AvgDiscount                0
FavoriteCategory           0
PreferredPayment           0
AvgPurchaseInterval    90594
dtype: int64

In [43]:
customer_features["AvgPurchaseInterval"] = (
    customer_features["AvgPurchaseInterval"]
    .fillna(0)
)

In [44]:
customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

In [45]:
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,AvgQuantity,AvgDiscount,FavoriteCategory,PreferredPayment,AvgPurchaseInterval
0,14,266,1,256.232791,5.0,15.504050,Books,Credit Card,0.0
1,42,345,1,502.656523,7.0,19.191090,Home Decor,PayPal,0.0
2,49,328,1,21.399047,1.0,14.923377,Electronics,Debit Card,0.0
3,59,27,2,249.492696,6.0,7.175628,Clothing,Debit Card,225.0
4,65,315,1,548.006625,8.0,8.319811,Clothing,PayPal,0.0


### RFM & Behavioral Features Table

| Column | Meaning | Business Interpretation |
|---|---|---|
| CustomerID | Unique customer identifier | Each row represents one customer |
| Recency | Days since last purchase | Lower value indicates more active customers |
| Frequency | Number of purchases | Higher value indicates loyal customers |
| Monetary | Total money spent | Higher value indicates high-value customers |
| AvgQuantity | Average products per order | Measures buying intensity |
| AvgDiscount | Average discount used | Indicates discount sensitivity |
| FavoriteCategory | Most purchased category | Shows product preference |
| PreferredPayment | Most used payment method | Represents payment behavior |
| AvgPurchaseInterval | Average days between purchases | Identifies buying cycle |